# Closed-Loop (Thin Colab Notebook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/thesis-research-colab/blob/main/notebooks/closedloop_simulation_colab.ipynb)

This notebook is intentionally thin. Core logic lives in `src/closedloop/*.py`.


## Notebook Purpose And Contract

This notebook is orchestration-only and keeps heavy logic in reusable Python modules.

Execution contract:
1. Sync latest code into `/content/thesis-research-colab`.
2. Mount Google Drive for persistent artifacts.
3. Bootstrap deterministic runtime only when actually needed.
4. Run diagnosis (quick probe + preflight + calibration gate).
5. Auto-run full optimization only if diagnostics are healthy.

Fast patch-and-diagnose loop (same Colab runtime):
1. Run Step 1 (repo sync) after each push.
2. Skip remount/setup if notebook reports cached-ready.
3. Re-run Step 4 onward; main loop will auto-run only when ready.



## Step 1 - Sync Repository

This cell keeps the runtime aligned with the current `main` branch.

Behavior:
- If repo directory already exists, it does `fetch + checkout main + pull --ff-only`.
- Otherwise it clones a fresh copy.

If this repo is private, you must authenticate git in Colab before this step.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/thesis-research-colab.git"
REPO_DIR = "/content/thesis-research-colab"
REPO_BRANCH = "main"


def _run(*args):
    subprocess.run(list(args), check=True)


def ensure_repo_ready(repo_url: str, repo_dir: str, branch: str = "main") -> str:
    repo_path = Path(repo_dir)
    content_root = Path('/content')
    content_root.mkdir(parents=True, exist_ok=True)

    # Recover if previous cwd was deleted (common after rm -rf on repo dir).
    try:
        _ = os.getcwd()
    except FileNotFoundError:
        os.chdir(str(content_root))

    if repo_path.exists() and (Path.cwd().resolve() == repo_path.resolve()):
        os.chdir(str(content_root))

    if repo_path.exists():
        print(f"[repo] existing checkout: {repo_path}")
        _run("git", "-C", str(repo_path), "fetch", "origin")
        _run("git", "-C", str(repo_path), "checkout", branch)
        _run("git", "-C", str(repo_path), "pull", "--ff-only", "origin", branch)
    else:
        print(f"[repo] cloning {repo_url} -> {repo_path}")
        _run("git", "clone", "--depth", "1", "-b", branch, repo_url, str(repo_path))

    if not (repo_path / 'src' / 'closedloop').exists():
        raise RuntimeError(f"[repo] missing expected module path: {repo_path / 'src' / 'closedloop'}")

    os.chdir(str(repo_path))
    repo_root = str(repo_path)
    src_root = str(repo_path / 'src')
    for path in (repo_root, src_root):
        if path not in sys.path:
            sys.path.insert(0, path)

    rev = subprocess.check_output(["git", "-C", str(repo_path), "rev-parse", "--short", "HEAD"], text=True).strip()
    return rev


rev = ensure_repo_ready(REPO_URL, REPO_DIR, REPO_BRANCH)
globals()['_CLOSEDLOOP_REPO_REV'] = str(rev)
print("Working directory:", os.getcwd())
print("Repo commit:", rev)
print("Has src/closedloop:", os.path.exists(os.path.join(REPO_DIR, "src", "closedloop")))
print("sys.path[0:3]:", sys.path[:3])



## Step 2 - Mount Persistent Storage

Artifacts such as checkpoints, per-scenario outputs, traces, and thresholds are written under `PERSIST_ROOT`.

Why this matters:
- Colab VM storage (`/content`) is ephemeral.
- Drive-mounted paths (`/content/drive/...`) let you resume runs after interruptions.


In [ ]:
# Auto-mount Google Drive for persistent resume artifacts (Colab only)
import os
import subprocess
import time
from pathlib import Path
from uuid import uuid4

REQUIRED_DRIVE_FOLDER = Path('/content/drive/MyDrive/waymax_experiments')
VERIFY_DRIVE_ACCESS_EVERY_RUN = False

if os.environ.get('COLAB_RELEASE_TAG'):
    from google.colab import auth, drive

    already_ready = bool(globals().get('_CLOSEDLOOP_DRIVE_READY', False))
    if already_ready and os.path.ismount('/content/drive') and REQUIRED_DRIVE_FOLDER.exists() and (not VERIFY_DRIVE_ACCESS_EVERY_RUN):
        print('[drive] already validated in this runtime; skipping remount/probe.')
    else:
        def _cleanup_stale_mount_state() -> None:
            subprocess.run(['bash', '-lc', 'fusermount -u /content/drive || true'], check=False)
            subprocess.run(['bash', '-lc', 'umount /content/drive || true'], check=False)
            os.makedirs('/content/drive', exist_ok=True)

        def _mount_with_retries() -> None:
            if os.path.ismount('/content/drive'):
                print('[drive] /content/drive already mounted')
                return

            errors = []
            attempts = [
                ('initial_mount', False, False, False),
                ('force_remount', True, False, False),
                ('auth_and_cleanup_force', True, True, True),
            ]
            for label, force_remount, do_auth, do_cleanup in attempts:
                try:
                    if do_auth:
                        auth.authenticate_user()
                    if do_cleanup:
                        _cleanup_stale_mount_state()
                    drive.mount('/content/drive', force_remount=force_remount)
                    if os.path.ismount('/content/drive'):
                        print(f'[drive] mount succeeded via {label}')
                        return
                    errors.append(f'{label}: mount returned without active mount')
                except Exception as e:
                    errors.append(f'{label}: {type(e).__name__}: {e}')
                    time.sleep(1.0)

            raise RuntimeError(
                '[drive] mount failed after retries. '
                + ' | '.join(errors)
                + ' ; If this persists: Runtime -> Disconnect and delete runtime, then rerun Step 2.'
            )

        _mount_with_retries()

        if not REQUIRED_DRIVE_FOLDER.exists():
            raise RuntimeError(
                '[drive] Required folder missing: /content/drive/MyDrive/waymax_experiments. '
                'Ask Achyut Morang (owner) for edit access, then rerun this cell.'
            )

        probe_file = REQUIRED_DRIVE_FOLDER / f'.codex_write_probe_{uuid4().hex}.tmp'
        try:
            probe_file.write_text('ok', encoding='utf-8')
            probe_file.unlink(missing_ok=True)
            print(f'[drive] Verified read/write access: {REQUIRED_DRIVE_FOLDER}')
        except Exception as e:
            raise RuntimeError(
                '[drive] Folder exists but is not writable: /content/drive/MyDrive/waymax_experiments. '
                'Ask Achyut Morang (owner) for edit access, then rerun this cell.'
            ) from e

        globals()['_CLOSEDLOOP_DRIVE_READY'] = True
else:
    print('[drive] non-Colab environment; skipping mount')



## Step 3 - Deterministic Runtime Setup

Setup is offloaded to `scripts/colab_setup.py`.

This cell is now runtime-cache aware:
- It reuses a ready setup within the same runtime session.
- It re-runs setup automatically only when needed (requirements fingerprint changes or core imports fail).
- It still restarts runtime when compiled dependencies were mutated.



In [ ]:
# Deterministic setup (runtime-cache aware)
from pathlib import Path
import importlib.util
import hashlib
import json
import os
import sys
import time

FORCE_REINSTALL = False
AUTO_RESTART_AFTER_SETUP = True
STRICT_LOCKFILE_CHECK = True
SETUP_CACHE_ENABLED = True
REVALIDATE_CORE_IMPORTS_ON_CACHE_HIT = True
SETUP_CACHE_PATH = Path('/content/.closedloop_setup_cache.json')

repo_root = Path.cwd()
setup_py = repo_root / "scripts" / "colab_setup.py"
req_path = repo_root / "requirements-colab.txt"
if not setup_py.exists():
    raise FileNotFoundError(f"Setup helper missing: {setup_py}")
if not req_path.exists():
    raise FileNotFoundError(f"Requirements file missing: {req_path}")

spec = importlib.util.spec_from_file_location("colab_setup", str(setup_py))
if spec is None or spec.loader is None:
    raise RuntimeError(f"Unable to load setup helper from {setup_py}")

colab_setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_setup)


def _setup_fingerprint() -> str:
    h = hashlib.sha256()
    h.update(req_path.read_bytes())
    h.update(str(sys.executable).encode('utf-8'))
    h.update(str(sys.version).encode('utf-8'))
    h.update(str(bool(STRICT_LOCKFILE_CHECK)).encode('utf-8'))
    return h.hexdigest()


def _core_import_probe() -> tuple[bool, str]:
    try:
        import jax  # noqa: F401
        import waymax  # noqa: F401
        import numpy as np  # noqa: F401
        import pandas as pd  # noqa: F401
        import scipy  # noqa: F401
        import sklearn  # noqa: F401
        from numpy._core.umath import _center, _expandtabs  # noqa: F401
        return True, f"ok numpy={np.__version__} pandas={pd.__version__}"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


setup_fingerprint = _setup_fingerprint()
setup_result = None
cache_hit = False
cache_reason = ''

cached = {}
if SETUP_CACHE_ENABLED and SETUP_CACHE_PATH.exists():
    try:
        cached = json.loads(SETUP_CACHE_PATH.read_text())
    except Exception:
        cached = {}

if SETUP_CACHE_ENABLED and (not FORCE_REINSTALL):
    if cached.get('fingerprint') == setup_fingerprint and cached.get('status') == 'ready':
        cache_hit = True
        cache_reason = 'fingerprint_match'

if cache_hit and REVALIDATE_CORE_IMPORTS_ON_CACHE_HIT:
    core_ok, core_msg = _core_import_probe()
    if not core_ok:
        cache_hit = False
        cache_reason = f'cache_invalid_core_probe_failed: {core_msg}'
    else:
        setup_result = {
            'ran_setup': False,
            'cache_hit': True,
            'cache_reason': cache_reason,
            'restart_required': False,
            'core_probe': core_msg,
            'kernel_executable': sys.executable,
        }
        print('[setup] cache hit -> skipping deterministic setup:', core_msg)

if setup_result is None:
    if cache_reason and (not cache_hit):
        print(f'[setup] cache bypassed: {cache_reason}')
    setup_result = colab_setup.run_deterministic_setup(
        force_reinstall=FORCE_REINSTALL,
        auto_restart=AUTO_RESTART_AFTER_SETUP,
        strict_lock=STRICT_LOCKFILE_CHECK,
    )

print("[setup] result:", setup_result)

if bool(setup_result.get('restart_required', False)):
    if not AUTO_RESTART_AFTER_SETUP:
        print("[setup] restart_required=True; please restart runtime before running simulation cells.")
else:
    if SETUP_CACHE_ENABLED:
        payload = {
            'status': 'ready',
            'fingerprint': setup_fingerprint,
            'timestamp_utc': int(time.time()),
            'kernel_executable': sys.executable,
            'repo_rev': str(globals().get('_CLOSEDLOOP_REPO_REV', 'unknown')),
            'strict_lock': bool(STRICT_LOCKFILE_CHECK),
        }
        SETUP_CACHE_PATH.write_text(json.dumps(payload, indent=2))
        print(f'[setup] cached ready state at {SETUP_CACHE_PATH}')



### Post-Restart Verification (Recommended)

If setup restarted runtime, rerun Steps 1-4 first.

Quick verification:

```python
import sys, numpy as np
print("kernel:", sys.executable)
print("numpy:", np.__version__, np.__file__)
from numpy._core.umath import _center, _expandtabs
print("NumPy private-symbol probe: OK")
```



### Optional Manual Checkpoint Fallback

Use this only if automatic checkpoint download fails.


In [ ]:
# Manual checkpoint fallback (run only if setup could not fetch checkpoint)
# !mkdir -p /content/checkpoints
# !wget -O /content/checkpoints/lantentdriver_t2_J3.ckpt https://huggingface.co/Sephirex-x/LatentDriver/resolve/main/checkpoints/lantentdriver_t2_J3.ckpt
# !ls -lh /content/checkpoints/lantentdriver_t2_J3.ckpt


## Step 4 - Import Modular Pipeline APIs

All experiment logic is imported from `src/closedloop` modules:
- configuration,
- calibration,
- search,
- export/resume utilities.


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_DIR = "/content/thesis-research-colab"
FORCE_MODULE_HOT_RELOAD = True
repo_path = Path(REPO_DIR)
if not repo_path.exists():
    raise RuntimeError("Repo checkout missing at /content/thesis-research-colab. Run the repo-sync cell first.")

os.chdir(str(repo_path))
for path in (str(repo_path), str(repo_path / 'src')):
    if path not in sys.path:
        sys.path.insert(0, path)

# Hot-reload safety after git pull.
if bool(FORCE_MODULE_HOT_RELOAD):
    for mod in [m for m in list(sys.modules) if (m == 'src' or m.startswith('src.'))]:
        sys.modules.pop(mod, None)
    importlib.invalidate_caches()

try:
    from src.closedloop import (
        analyze_signal_if_available,
        auto_select_shard_id,
        configure_persistent_run_prefix,
        initialize_configs,
        inspect_shard_progress,
        run_main_loop_with_policy,
        run_preflight_bundle,
        run_quick_probe_with_auto_escalation,
        run_surprise_gate_with_policy,
        summarize_and_export_if_available,
    )
except ModuleNotFoundError as e:
    msg = (
        "Import failed for src.closedloop. Re-run repo-sync cell, ensure cwd=/content/thesis-research-colab, then rerun this cell.\n"
        f"cwd={os.getcwd()}\n"
        f"sys.path_head={sys.path[:5]}"
    )
    raise RuntimeError(msg) from e


## Step 5 - Configure Run Identity, Persistence, And Auto-Run Policy

Key controls:
- `RUN_TAG`: experiment identifier.
- `PERSIST_ROOT`: persistent artifact root.
- `N_SHARDS` and `SHARD_ID`: run slicing and resume behavior.
- `AUTO_RUN_MAIN_LOOP_WHEN_READY`: automatically run simulation when diagnostics pass.
- `RUN_MAIN_LOOP_OVERRIDE`: optional override (`None`/`True`/`False`).

Recommended dev loop:
- keep auto-run enabled,
- let diagnostics decide if full simulation should execute.



In [ ]:
cfg, search_cfg, ckpt_scan_df = initialize_configs()

# Auto-run policy for full simulation:
# - None: follow diagnostics readiness
# - True: force run (if run cell is reached)
# - False: force skip
RUN_MAIN_LOOP_OVERRIDE = None
AUTO_RUN_MAIN_LOOP_WHEN_READY = True

# Fast-iteration profile.
cfg.n_eval_scenarios = 100
cfg.strict_min_eval = 100
cfg.n_total_scenarios = max(cfg.n_total_scenarios, 400)

# LatentDriver robustness knobs.
cfg.latentdriver_auto_align_token_count = True
cfg.latentdriver_log_forward_errors = True
cfg.latentdriver_log_forward_errors_max = 10

RUN_TAG = "closedloop_v1"
PERSIST_ROOT = "/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1"
N_SHARDS = 5
SHARD_ID = "auto"   # set int to force a specific shard; use "auto" to resume next shard automatically
RESTORE_FROM_UPLOAD = False

shard_progress_df = inspect_shard_progress(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
)

if isinstance(SHARD_ID, str) and SHARD_ID.strip().lower() == "auto":
    SHARD_ID = auto_select_shard_id(
        run_tag=RUN_TAG,
        persist_root=PERSIST_ROOT,
        n_shards=N_SHARDS,
    )
    print(f"[shard] auto-selected SHARD_ID={SHARD_ID}")
else:
    SHARD_ID = int(SHARD_ID)

run_prefix = configure_persistent_run_prefix(
    cfg=cfg,
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    shard_id=SHARD_ID,
    n_shards=N_SHARDS,
)
print("run_prefix =", run_prefix)
print(f"[shard] running {SHARD_ID + 1}/{max(1, N_SHARDS)}")
print(f"[run-policy] AUTO_RUN_MAIN_LOOP_WHEN_READY={AUTO_RUN_MAIN_LOOP_WHEN_READY}, RUN_MAIN_LOOP_OVERRIDE={RUN_MAIN_LOOP_OVERRIDE}")

if len(shard_progress_df):
    display(shard_progress_df)

if len(ckpt_scan_df):
    display(ckpt_scan_df.head(10))



## Step 6 - Build Dataset, Splits, And Scenario Handles

This stage:
- validates WOMD GCS access,
- optionally runs a tiny early surprise-probe on a few WOMD scenarios,
- builds reference/evaluation splits,
- retains simulator states for eval scenarios,
- prepares reference/open-loop baselines.


In [ ]:
RUN_QUICK_SURPRISE_PROBE = True
QUICK_PROBE_SCENARIOS = 1
QUICK_PROBE_PROPOSALS_PER_SCENARIO = 4
STOP_IF_QUICK_PROBE_COLLAPSED = False

# Auto-escalate only if the 1-scenario probe collapses.
AUTO_ESCALATE_QUICK_PROBE = True
MAX_PROBE_ESCALATIONS = 3
PROBE_SCALE_MULTIPLIERS = (1.0, 1.35, 1.8)
PROBE_DELTA_L2_MULTIPLIERS = (1.0, 1.2, 1.4)
PROBE_DELTA_CLIP_MULTIPLIERS = (1.0, 1.1, 1.2)
PROBE_BUDGET_BUMP_PER_ESCALATION = 2
APPLY_SUCCESSFUL_PROBE_TUNING = True

probe_bundle = run_quick_probe_with_auto_escalation(
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    run_quick_surprise_probe_enabled=bool(RUN_QUICK_SURPRISE_PROBE),
    quick_probe_scenarios=int(QUICK_PROBE_SCENARIOS),
    quick_probe_proposals_per_scenario=int(QUICK_PROBE_PROPOSALS_PER_SCENARIO),
    stop_if_quick_probe_collapsed=bool(STOP_IF_QUICK_PROBE_COLLAPSED),
    auto_escalate_quick_probe=bool(AUTO_ESCALATE_QUICK_PROBE),
    max_probe_escalations=int(MAX_PROBE_ESCALATIONS),
    probe_scale_multipliers=tuple(PROBE_SCALE_MULTIPLIERS),
    probe_delta_l2_multipliers=tuple(PROBE_DELTA_L2_MULTIPLIERS),
    probe_delta_clip_multipliers=tuple(PROBE_DELTA_CLIP_MULTIPLIERS),
    probe_budget_bump_per_escalation=int(PROBE_BUDGET_BUMP_PER_ESCALATION),
    apply_successful_probe_tuning=bool(APPLY_SUCCESSFUL_PROBE_TUNING),
)

cfg = probe_bundle.cfg
search_cfg = probe_bundle.search_cfg
quick_probe_df = probe_bundle.quick_probe_df
quick_probe_summary_df = probe_bundle.quick_probe_summary_df
quick_probe_attempts_df = probe_bundle.quick_probe_attempts_df

dataset_config = probe_bundle.dataset_config
data_iter = probe_bundle.data_iter
runner = probe_bundle.runner
data = probe_bundle.data
reference_idx = probe_bundle.reference_idx
candidate_eval_idx = probe_bundle.candidate_eval_idx
eval_idx_all = probe_bundle.eval_idx_all
eval_idx = probe_bundle.eval_idx
reference_df = probe_bundle.reference_df
base_eval_openloop_df = probe_bundle.base_eval_openloop_df

if len(quick_probe_attempts_df):
    display(quick_probe_attempts_df)
if len(quick_probe_summary_df):
    display(quick_probe_summary_df)
if len(quick_probe_df):
    display(quick_probe_df.head(20))

if probe_bundle.applied_tuning:
    print('[probe] applied tuned search settings from successful attempt.')
    print('        proposal_scale_ladder=', search_cfg.proposal_scale_ladder)
    print('        delta_l2_budget=', search_cfg.delta_l2_budget)
    print('        delta_clip=', search_cfg.delta_clip)
    print('        budget_evals=', search_cfg.budget_evals)

if probe_bundle.final_collapsed:
    print('[probe] warning: quick probe remained collapsed after escalation attempts.')


In [ ]:
print(f"reference={len(reference_idx)} eval_pool={len(candidate_eval_idx)} eval_shard={len(eval_idx)}")



## Step 7 - Preflight Checks And Closed-Loop Calibration

Preflight verifies planner and rollout health before expensive simulation.
Calibration estimates risk/surprise thresholds and scaling used by search objectives.


In [ ]:
STOP_ON_PREFLIGHT_FAIL = False

preflight_bundle = run_preflight_bundle(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=RESTORE_FROM_UPLOAD,
    stop_on_preflight_fail=bool(STOP_ON_PREFLIGHT_FAIL),
)

READY_FOR_MAIN_LOOP = preflight_bundle.ready_for_main_loop
preflight_df = preflight_bundle.preflight_df
closedloop_calib_df = preflight_bundle.closedloop_calib_df
closedloop_thresholds = preflight_bundle.closedloop_thresholds
calib_diag_df = preflight_bundle.calib_diag_df
calib_quant_df = preflight_bundle.calib_quant_df
root_cause_summary_df = preflight_bundle.root_cause_summary_df
root_cause_findings_df = preflight_bundle.root_cause_findings_df

if preflight_bundle.preflight_error and (not READY_FOR_MAIN_LOOP):
    print('[preflight] failed; calibration/main loop will be skipped until checks pass.')
    print(preflight_bundle.preflight_error)
if preflight_bundle.root_cause_diag_error:
    print(f"[diagnose] root-cause analysis failed: {preflight_bundle.root_cause_diag_error}")

print('READY_FOR_MAIN_LOOP =', READY_FOR_MAIN_LOOP)
if len(preflight_df):
    display(preflight_df)
if len(root_cause_summary_df):
    display(root_cause_summary_df)
if len(root_cause_findings_df):
    display(root_cause_findings_df)
if READY_FOR_MAIN_LOOP:
    print('Calibration thresholds:', closedloop_thresholds)
    display(calib_diag_df)
    display(calib_quant_df)
else:
    print('[preflight] Fix failing checks, then rerun this cell.')


## Step 8 - Surprise Quality Gate

This gate prevents running full optimization when surprise signal quality is degenerate
(for example near-zero variance or excessive fallback behavior).


In [ ]:
SURPRISE_GATE_ENABLED = True
STOP_ON_GATE_FAIL = False
ALLOW_MAIN_LOOP_WHEN_GATE_FAILS = False

gate_bundle = run_surprise_gate_with_policy(
    closedloop_calib_df=closedloop_calib_df,
    ready_for_main_loop=bool(globals().get('READY_FOR_MAIN_LOOP', False)),
    surprise_gate_enabled=bool(SURPRISE_GATE_ENABLED),
    stop_on_gate_fail=bool(STOP_ON_GATE_FAIL),
    allow_main_loop_when_gate_fails=bool(ALLOW_MAIN_LOOP_WHEN_GATE_FAILS),
)

READY_FOR_OPTIMIZATION = gate_bundle.ready_for_optimization
gate_summary_df = gate_bundle.gate_summary_df
dist_change_summary_df = gate_bundle.dist_change_summary_df

if gate_bundle.gate_skipped_reason:
    print(f"[gate] skipped: {gate_bundle.gate_skipped_reason}")
if gate_bundle.gate_error:
    print('[gate] failed:', gate_bundle.gate_error)

print('READY_FOR_OPTIMIZATION =', READY_FOR_OPTIMIZATION)
if len(gate_summary_df):
    display(gate_summary_df)
if len(dist_change_summary_df):
    display(dist_change_summary_df)


## Step 9 - Closed-Loop Optimization Run (Auto-Gated)

This step auto-runs only when diagnostics set `READY_FOR_OPTIMIZATION=True`.
Use `RUN_MAIN_LOOP_OVERRIDE` only when you explicitly want to force run/skip.



In [ ]:
main_loop_bundle = run_main_loop_with_policy(
    runner=runner,
    eval_idx=eval_idx,
    cfg=cfg,
    search_cfg=search_cfg,
    closedloop_thresholds=closedloop_thresholds,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    ready_for_optimization=bool(globals().get('READY_FOR_OPTIMIZATION', False)),
    auto_run_main_loop_when_ready=bool(globals().get('AUTO_RUN_MAIN_LOOP_WHEN_READY', True)),
    run_main_loop_override=globals().get('RUN_MAIN_LOOP_OVERRIDE', None),
)

should_run_main_loop = main_loop_bundle.should_run_main_loop
policy = main_loop_bundle.policy
closedloop_results_df = main_loop_bundle.closedloop_results_df
closedloop_trace_df = main_loop_bundle.closedloop_trace_df

print(f'[run-policy] {policy} -> should_run_main_loop={should_run_main_loop}')
if not should_run_main_loop:
    if not bool(globals().get('READY_FOR_OPTIMIZATION', False)):
        print('[run] skipped: diagnostics not ready (READY_FOR_OPTIMIZATION=False).')
    else:
        print('[run] skipped by policy override/auto-run setting.')

print('Result rows:', len(closedloop_results_df))
print('Trace rows:', len(closedloop_trace_df))


## Step 10 - Summarize And Export Artifacts

Exports include per-scenario outputs, traces, thresholds, diagnostics, and runtime manifest.
These files are the primary inputs for paper tables/plots and reproducibility records.


In [ ]:
export_bundle = summarize_and_export_if_available(
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    closedloop_results_df=closedloop_results_df,
    closedloop_trace_df=closedloop_trace_df,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    closedloop_thresholds=closedloop_thresholds,
)

quick_summary_df = export_bundle.quick_summary_df
sanity_df = export_bundle.sanity_df
fairness_checks_df = export_bundle.fairness_checks_df
trace_diag_df = export_bundle.trace_diag_df
artifact_paths = export_bundle.artifact_paths

if closedloop_results_df.empty:
    print('[export] skipped: closedloop_results_df is empty (main loop was skipped or produced no rows).')

if len(quick_summary_df):
    display(quick_summary_df)
if len(sanity_df):
    display(sanity_df)
if len(fairness_checks_df):
    display(fairness_checks_df)
if len(trace_diag_df):
    display(trace_diag_df)

if artifact_paths:
    print('Artifacts:')
    for key, value in artifact_paths.items():
        print(f' - {key}: {value}')


## Step 11 - Surprise Signal Usefulness Diagnostics

Check whether `surprise_pd` is actually informative for optimization outcomes (`delta_risk`).
This computes global correlations, quantile-bin monotonicity, top-k lift, and within-scenario rank consistency.


In [ ]:
signal_bundle = analyze_signal_if_available(
    closedloop_results_df=closedloop_results_df,
    n_bins=10,
    top_fracs=(0.10, 0.20),
    scenario_min_points=3,
)

signal_summary_df = signal_bundle.signal_summary_df
signal_method_corr_df = signal_bundle.signal_method_corr_df
signal_bin_df = signal_bundle.signal_bin_df
signal_topk_lift_df = signal_bundle.signal_topk_lift_df
signal_within_scenario_df = signal_bundle.signal_within_scenario_df

if closedloop_results_df.empty:
    print('[signal] skipped: closedloop_results_df is empty.')

if len(signal_summary_df):
    display(signal_summary_df)
if len(signal_method_corr_df):
    display(signal_method_corr_df)
if len(signal_bin_df):
    display(signal_bin_df)
if len(signal_topk_lift_df):
    display(signal_topk_lift_df)
if len(signal_within_scenario_df):
    display(signal_within_scenario_df.head(20))
    if len(signal_within_scenario_df) > 20:
        print(f'[signal] showing first 20 rows of {len(signal_within_scenario_df)} within-scenario rows.')
